# Importing Libraries 

In [3]:
import pandas as pd
import ast
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Reading dataset

In [16]:
df = pd.read_csv('updated_aggregated_advertisers_dataset_v2.csv')

# Cleaning industry column data

In [17]:
def parse_industry(val):
    try:
        # Convert string like "['Home & Garden']" into a real list
        parsed = ast.literal_eval(val)
        # Join list items into a single string
        return ', '.join(parsed).lower()
    except:
        # If something goes wrong (bad format), return as is
        return val

# Cleaning goals column data

In [18]:
def clean_goals(val):
    try:
        # Step 1: Convert outer string into actual list
        outer_list = ast.literal_eval(val)
        if isinstance(outer_list, list) and len(outer_list) > 0:
            inner_str = outer_list[0]
            # Step 2: Extract goal keywords (e.g., AWARENESS, LEADS)
            goals = re.findall(r"[A-Z_]+", inner_str)
            # Step 3: Join and lowercase
            return ", ".join(goals).lower()
        return ""
    except Exception:
        return val  # fallback if broken format

# Applying cleaning functions to unique_industries & unique_goals column

In [19]:
df['unique_industries'] = df['unique_industries'].apply(parse_industry)
df['unique_goals'] = df['unique_goals'].apply(clean_goals)

# Just lower casing the entire data frame string values

In [20]:
df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

/var/folders/bl/c0pvtpr55y3477p6dl_6s5kh0000gn/T/ipykernel_4998/2340421553.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


# Applying literal_eval to postal codes column

In [21]:
df['postal_codes'] = df['postal_codes'].apply(ast.literal_eval)

# Dropping the columns which will not be used further

In [22]:
df = df.drop('Unnamed: 0', axis=1)
df = df.drop(columns=["order_goals", "unique_order_count", "avg_order_duration_days", "states", "locations", "cities",
 'line_item_ids',
 'unique_order_count',
 'avg_order_duration_days',
 'states',
 'locations',
 'cities',
 'allocation_goal_types',
 'avg_line_item_duration_days',
 'total_successes',
 'allocation_ordered_ad_spend_sum',
 'advertiser_favorite_postal_code',
 'success_ratio',
 'total_entries',
 'favorite_city',
 'favorite_state',
 'unique_line_item_count',
 'industry_count',
 'is_focused_on_1_industry',
 'state_count',
 'is_focused_on_1_state',
 'city_count',
 'is_focused_on_1_city',
 'postal_code_count',
 'is_focused_on_1_postal_code',
 'advertiser_postal_code',
 'advertiser_city',
 'advertiser_state_code',
 'runs_in_business_state',
 'runs_in_business_city'])

# for performance metric checking goals that are empty

In [23]:
empty_strings = (df['unique_goals'] == '').sum()
print(f"Empty strings (''): {empty_strings}")

Empty strings (''): 48


# Creating df of the empty goals with other columns 

In [24]:
empty_goals_mask = df['unique_goals'] == ''
empty_goal_rows = df[empty_goals_mask]
columns_to_show = ['advertiser_name', 'unique_industries', 'postal_codes', 'unique_goals']
customized_df = pd.DataFrame(empty_goal_rows[columns_to_show])


# Creating pickle file

In [178]:
df.to_pickle('final_preprocessed_df.pkl')

# Load pickle file

In [4]:
final_preprocessed_df = pd.read_pickle('final_preprocessed_df.pkl')

# columns_to_use for entire modeling 

In [5]:
columns_to_use = ['unique_goals', 'unique_industries', 'advertiser_name', 'postal_codes']

# Preprocessing column data for vectorization

In [6]:
def preprocess_text_preserve_multiword(series):
    return series.astype(str) \
        .str.replace(r"[\[\]']", "", regex=True) \
        .str.replace(r"[&]", "", regex=True) \
        .str.split(",") \
        .apply(lambda items: ' '.join(
            re.sub(r'[^\w\s]', '', item).strip().replace(" ", "_")
            for item in items if item.strip()
        ))

# Orignal df vectorization for recommendation using BOW

In [7]:
def create_semantic_bow_vectors(df):
    goal_cols = ['unique_goals']
    industry_cols = ['unique_industries']
    advertiser_name_cols = ['advertiser_name']
    
    vectors = {}
    vectorizers = {}
    
    # 1. Goals BOW Vector
    goal_text = df[goal_cols].apply(preprocess_text_preserve_multiword)
    goal_text_combined = goal_text.apply(lambda row: ' '.join(row), axis=1)
    goal_vectorizer = CountVectorizer(lowercase=True)
    goal_bow = goal_vectorizer.fit_transform(goal_text_combined)
    goal_features = [f"{name}" for name in goal_vectorizer.get_feature_names_out()]
    vectors['goals'] = pd.DataFrame(goal_bow.toarray(), columns=goal_features, index=df.index)
    vectorizers['goals'] = goal_vectorizer
    
    # 2. Industry BOW Vector
    industry_text = df[industry_cols].apply(preprocess_text_preserve_multiword)
    industry_text_combined = industry_text.apply(lambda row: ' '.join(row), axis=1)
    industry_vectorizer = CountVectorizer(lowercase=True)
    industry_bow = industry_vectorizer.fit_transform(industry_text_combined)
    industry_features = [f"{name}" for name in industry_vectorizer.get_feature_names_out()]
    vectors['industry'] = pd.DataFrame(industry_bow.toarray(), columns=industry_features, index=df.index)
    vectorizers['industry'] = industry_vectorizer
    
    # 3. Advertiser BOW Vector
    advertiser_text = df[advertiser_name_cols].apply(preprocess_text_preserve_multiword)
    advertiser_text_combined = advertiser_text.apply(lambda row: ' '.join(row), axis=1)
    advertiser_vectorizer = CountVectorizer(lowercase=True)
    advertiser_bow = advertiser_vectorizer.fit_transform(advertiser_text_combined)
    advertiser_features = [f"{name}" for name in advertiser_vectorizer.get_feature_names_out()]
    vectors['advertiser'] = pd.DataFrame(advertiser_bow.toarray(), columns=advertiser_features, index=df.index)
    vectorizers['advertiser'] = advertiser_vectorizer

    return vectors, vectorizers

# Create semantic vectors
semantic_vectors, semantic_vectorizers = create_semantic_bow_vectors(final_preprocessed_df)

# print("Geographic BOW shape:", semantic_vectors['geographic'].shape)
print("Goals BOW shape:", semantic_vectors['goals'].shape) 
print("Industry BOW shape:", semantic_vectors['industry'].shape)
print("Advertiser BOW shape:", semantic_vectors['advertiser'].shape)

# Option 1: Use vectors separately for different analyses
# geo_similarity = semantic_vectors['geographic']
goal_similarity = semantic_vectors['goals']
industry_similarity = semantic_vectors['industry']
advertiser_similarity = semantic_vectors['advertiser']

# Option 2: Combine all vectors (if needed for single model)
combined_semantic_bow = pd.concat([
    # semantic_vectors['geographic'],
    semantic_vectors['goals'], 
    semantic_vectors['industry'],
    semantic_vectors['advertiser']
], axis=1)

combined_semantic_bow_with_advertiser_name = pd.concat([
    final_preprocessed_df[['advertiser_name']],
    combined_semantic_bow
], axis=1)

combined_semantic_bow = combined_semantic_bow.reset_index(drop=True)

Goals BOW shape: (1390, 5)
Industry BOW shape: (1390, 30)
Advertiser BOW shape: (1390, 1333)


In [8]:
combined_semantic_bow_with_advertiser_name

,advertiser_name,awareness,engagement,foot_traffic,leads,retention,agriculture,apparel__accessories,arts__entertainment,autos__vehicles,...,zamzow_mfg,zatarains,zen_windows_boston,zen_windows_cleveland,zen_windows_philadelphia,zen_windows_st_louis,zen_windows_twin_cities,zen_windows_west_michigan,zeta_charter_schools,zoofari_parks
0,n fox jewelers,1,1,1,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1 800 plumber cypress,1,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1 800 plumber lansdale,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1 800 plumber monterey,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,180 water,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1385,dataparc,1,1,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1386,ilearn schools,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1387,ilearn schools - bronx,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1388,lake chelan health,1,1,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# User side vectorization and recommending location by using cosine similarity. 

In [9]:
class LocationRecommendationSystem:
    def __init__(self, semantic_vectorizers, feature_matrix, original_df):
        self.semantic_vectorizers = semantic_vectorizers
        self.feature_matrix = feature_matrix  
        self.original_df = original_df
    
    def build_user_input_record(self, advertiser_name, industry, goals):
        advertiser_name_cleaned = advertiser_name if advertiser_name else 'unknown_advertiser'
        industry_cleaned = industry if industry else 'unknown_industry'
        goals_cleaned = goals if goals else 'unknown_goals'

        return pd.DataFrame([{
            'unique_goals': goals_cleaned,
            'unique_industries': industry_cleaned,
            'advertiser_name': advertiser_name_cleaned
        }])
    
    def create_semantic_bow_vectors_inference(self, user_df):
        """Transform user input using pre-fitted vectorizers."""
        semantic_vectorizers = self.semantic_vectorizers
        
        goal_cols = ['unique_goals']
        industry_cols = ['unique_industries']
        advertiser_name_cols = ['advertiser_name']
        
        vectors = {}
        
        # Goals BOW Vector
        goal_text = user_df[goal_cols].apply(preprocess_text_preserve_multiword)
        goal_text_combined = goal_text.apply(lambda row: ' '.join(row), axis=1)
        goal_bow = semantic_vectorizers['goals'].transform(goal_text_combined)
        goal_features = [f"{name}" for name in semantic_vectorizers['goals'].get_feature_names_out()]
        vectors['goals'] = pd.DataFrame(goal_bow.toarray(), columns=goal_features, index=user_df.index)
        
        # Industry BOW Vector
        industry_text = user_df[industry_cols].apply(preprocess_text_preserve_multiword)
        industry_text_combined = industry_text.apply(lambda row: ' '.join(row), axis=1)
        industry_bow = semantic_vectorizers['industry'].transform(industry_text_combined)
        industry_features = [f"{name}" for name in semantic_vectorizers['industry'].get_feature_names_out()]
        vectors['industry'] = pd.DataFrame(industry_bow.toarray(), columns=industry_features, index=user_df.index)
        
        # Advertiser BOW Vector
        advertiser_text = user_df[advertiser_name_cols].apply(preprocess_text_preserve_multiword)
        advertiser_text_combined = advertiser_text.apply(lambda row: ' '.join(row), axis=1)
        advertiser_bow = semantic_vectorizers['advertiser'].transform(advertiser_text_combined)
        advertiser_features = [f"{name}" for name in semantic_vectorizers['advertiser'].get_feature_names_out()]
        vectors['advertiser'] = pd.DataFrame(advertiser_bow.toarray(), columns=advertiser_features, index=user_df.index)
        
        return vectors
    
    def vectorize_user_input(self, user_df):
        # Create semantic vectors
        semantic_vectors = self.create_semantic_bow_vectors_inference(user_df)
        
        # Combine semantic vectors in the same order as training
        combined_semantic_bow = pd.concat([
            semantic_vectors['goals'],
            semantic_vectors['industry'],
            semantic_vectors['advertiser']
        ], axis=1)
        
        # Reset indices and combine all features
        combined_semantic_bow = combined_semantic_bow.reset_index(drop=True)

        return combined_semantic_bow.values

    def find_exact_matches(self, user_input_df):
        user_goals = user_input_df['unique_goals'].iloc[0]
        user_industry = user_input_df['unique_industries'].iloc[0]
        user_advertiser = user_input_df['advertiser_name'].iloc[0]

        df = self.original_df

        # Build boolean masks for each condition
        goals_mask = df['unique_goals'] == user_goals
        industry_mask = df['unique_industries'] == user_industry
        advertiser_mask = df['advertiser_name'] == user_advertiser

        # Combine all conditions
        all_conditions = goals_mask & industry_mask & advertiser_mask

        # Get indices where all conditions are True
        exact_matches = df[all_conditions].index.tolist()

        return exact_matches
    
    def get_similarity_recommendations(self, user_vector, top_k=5):
        # FIXED: Use feature_matrix instead of semantic_vectorizers
        similarities = cosine_similarity(user_vector, self.feature_matrix)[0]
        top_indices = similarities.argsort()[::-1][:top_k]
        
        recommendations = self.original_df.iloc[top_indices]['postal_codes'].copy()
        similarity_scores = similarities[top_indices]
        
        return recommendations, similarity_scores
    
    def recommend_locations(self, advertiser_name, industry, goals, top_k=5, prefer_exact_match=True):
        # Build user input record
        user_input_df = self.build_user_input_record(advertiser_name, industry, goals)

        # Try exact matching first
        if prefer_exact_match:
            exact_matches = self.find_exact_matches(user_input_df)
            
            if len(exact_matches) > 0:
                recommendations = self.original_df.iloc[exact_matches]['postal_codes'].head(top_k)
                all_postal_codes = []
                for codes in recommendations:
                    if isinstance(codes, list):
                        all_postal_codes.extend(codes)
                    else:
                        all_postal_codes.append(codes)

                # Remove duplicates while preserving order
                unique_postal_codes = []
                seen = set()
                for code in all_postal_codes:
                    if code not in seen:
                        unique_postal_codes.append(code)
                        seen.add(code)

                # Select top_k unique postal codes
                top_k_postal_codes = unique_postal_codes[:top_k]

                return {
                    'recommendations': top_k_postal_codes,
                    'method_used': 'exact_match',
                    'num_exact_matches': len(exact_matches),
                    'similarity_scores': None
                }
        
        # Fall back to similarity-based recommendations
        user_vector = self.vectorize_user_input(user_input_df)
        recommendations, similarity_scores = self.get_similarity_recommendations(user_vector, top_k=top_k)

        # Collect and flatten postal codes from top rows
        all_postal_codes = []
        for codes in recommendations:
            if isinstance(codes, list):
                all_postal_codes.extend(codes)
            else:
                all_postal_codes.append(codes)

        # Remove duplicates while preserving order
        unique_postal_codes = []
        seen = set()
        for code in all_postal_codes:
            if code not in seen:
                unique_postal_codes.append(code)
                seen.add(code)

        # Select top_k unique postal codes
        top_k_postal_codes = unique_postal_codes[:top_k]

        return {
            'recommendations': top_k_postal_codes,
            'method_used': 'similarity_based',
            'num_exact_matches': 0,
            'similarity_scores': similarity_scores
        }


# CORRECTED INITIALIZATION - Include the feature matrix
recommendation_system = LocationRecommendationSystem(
    semantic_vectorizers=semantic_vectorizers,      
    feature_matrix=combined_semantic_bow.values,    
    original_df=final_preprocessed_df              
)

## giving input to get recommendation


In [10]:
# Get recommendations
results = recommendation_system.recommend_locations(
    advertiser_name="n fox jewelers",
    industry='',
    goals="AWARENESS, ENGAGEMENT, FOOT_TRAFFIC, LEADS, RETENTION",
    top_k=6
)

In [11]:
results['recommendations']

['12871', '12065', '12183', '12866', '12884', '12222']

# Checking performance 
- Copies the first 100 rows of the original DataFrame.
- Adds a new column to store recommended postal codes.
- Loops through each row to extract advertiser info and goals.
- Sets top_k to the number of actual postal codes in that row.
- Generates and stores recommendations using the recommendation system.


In [27]:
df_with_recommendations = final_preprocessed_df.copy()

# Create a column to store the top recommended postal code(s)
df_with_recommendations['recommended_postal_code'] = ''

# Loop through each row
for idx, row in df_with_recommendations.iterrows():
    advertiser_name = row['advertiser_name'] if 'advertiser_name' in row else ''
    industry = row['unique_industries'] if 'unique_industries' in row else ''
    goals = row['unique_goals'] if 'unique_goals' in row else ''

    actual_postal_codes = row.get('postal_codes', [])
    top_k = len(actual_postal_codes) if isinstance(actual_postal_codes, list) else 5  # fallback to 5

    # Get recommendations
    result = recommendation_system.recommend_locations(
        advertiser_name=advertiser_name,
        industry=industry,
        goals=goals,
        top_k=top_k
    )

    recommended_codes = result['recommendations']  # ✅ fixed variable name
    
    if len(recommended_codes) > 0:
        df_with_recommendations.at[idx, 'recommended_postal_code'] = recommended_codes
    else:
        df_with_recommendations.at[idx, 'recommended_postal_code'] = ''

In [30]:
df_with_recommendations['recommended_postal_code']

0       [12871, 12065, 12183, 12866, 12884, 12222, 120...
1       [77388, 77433, 77386, 77373, 77429, 77380, 773...
2       [19002, 19004, 19456, 19446, 19436, 19405, 190...
3       [95066, 95004, 95073, 95007, 95019, 93905, 950...
4       [59018, 59720, 59739, 59450, 82423, 59019, 590...
                              ...                        
1385    [30240, 28070, 55949, 51543, 97016, 37744, 473...
1386           [07099, 07501, 07306, 07032, 07029, 07086]
1387    [10462, 10460, 10465, 10469, 10475, 10470, 104...
1388    [98841, 98830, 98813, 98846, 98840, 98833, 988...
1389    [98126, 98005, 98029, 98195, 98027, 98039, 980...
Name: recommended_postal_code, Length: 1390, dtype: object

### using the empty goals df we created named as: customized_df

In [32]:
df_with_empty_goals_recommendations = customized_df.copy()

In [42]:
# Create a column to store the top recommended postal code(s)
df_with_empty_goals_recommendations['recommended_postal_code'] = ''

# Loop through each row
for idx, row in df_with_empty_goals_recommendations.iterrows():
    advertiser_name = row['advertiser_name'] if 'advertiser_name' in row else ''
    industry = row['unique_industries'] if 'unique_industries' in row else ''
    goals = row['unique_goals'] if 'unique_goals' in row else ''

    actual_postal_codes = row.get('postal_codes', [])
    top_k = len(actual_postal_codes) if isinstance(actual_postal_codes, list) else 5  # fallback to 5

    # Get recommendations
    result = recommendation_system.recommend_locations(
        advertiser_name=advertiser_name,
        industry=industry,
        goals=goals,
        top_k=top_k
    )

    recommended_codes = result['recommendations']  # ✅ fixed variable name
    
    if len(recommended_codes) > 0:
        df_with_empty_goals_recommendations.at[idx, 'recommended_postal_code'] = recommended_codes
    else:
        df_with_empty_goals_recommendations.at[idx, 'recommended_postal_code'] = ''

In [43]:
df_with_empty_goals_recommendations['recommended_postal_code'].head()

16    [45429, 45845, 45381, 45359, 45404, 45324, 453...
18    [33610, 33810, 33604, 33613, 33617, 33511, 336...
24    [67360, 74004, 67340, 74051, 74001, 74072, 740...
31    [45419, 45429, 45068, 45383, 45404, 45324, 453...
33    [76692, 76179, 76059, 76034, 76107, 76155, 760...
Name: recommended_postal_code, dtype: object

# Perfomance metrices 

## Metric 01: matching the postal codes with recommended postal codes

In [201]:
def all_recommended_in_postal_codes(row):
    recommended = row['recommended_postal_code']
    actual = row['postal_codes']
    
    if isinstance(recommended, list) and isinstance(actual, list):
        return all(code in actual for code in recommended)
    return False

df_with_recommendations['is_match'] = df_with_recommendations.apply(all_recommended_in_postal_codes, axis=1)


## giving accuracy in percentage

In [202]:
accuracy = df_with_recommendations['is_match'].mean()
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 100.00%


## Metric 02: matching the postal codes with recommended postal codes of empty goals df

In [203]:
def all_recommended_in_postal_codes(row):
    recommended = row['recommended_postal_code']
    actual = row['postal_codes']
    
    if isinstance(recommended, list) and isinstance(actual, list):
        return all(code in actual for code in recommended)
    return False

df_with_empty_goals_recommendations['is_match'] = df_with_empty_goals_recommendations.apply(all_recommended_in_postal_codes, axis=1)


## giving accuracy in percentage

In [204]:
accuracy = df_with_empty_goals_recommendations['is_match'].mean()
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 100.00%


# applying python doc concepts

### without multiprocessing 

In [44]:
import time
start_time = time.time()
# Create a column to store the top recommended postal code(s)
df_with_empty_goals_recommendations['recommended_postal_code'] = ''

# Loop through each row
for idx, row in df_with_empty_goals_recommendations.iterrows():
    advertiser_name = row['advertiser_name'] if 'advertiser_name' in row else ''
    industry = row['unique_industries'] if 'unique_industries' in row else ''
    goals = row['unique_goals'] if 'unique_goals' in row else ''

    actual_postal_codes = row.get('postal_codes', [])
    top_k = len(actual_postal_codes) if isinstance(actual_postal_codes, list) else 5  # fallback to 5

    # Get recommendations
    result = recommendation_system.recommend_locations(
        advertiser_name=advertiser_name,
        industry=industry,
        goals=goals,
        top_k=top_k
    )

    recommended_codes = result['recommendations']  # ✅ fixed variable name
    
    if len(recommended_codes) > 0:
        df_with_empty_goals_recommendations.at[idx, 'recommended_postal_code'] = recommended_codes
    else:
        df_with_empty_goals_recommendations.at[idx, 'recommended_postal_code'] = ''

In [45]:
df_with_empty_goals_recommendations['recommended_postal_code'].head()
print(f"⏱️ Time taken without multiprocessing: {time.time() - start_time:.2f} seconds")

⏱️ Time taken without multiprocessing: 4.35 seconds


### with multiprocessing 

In [47]:
def process_row(row):
    advertiser_name = row.get('advertiser_name', '')
    industry = row.get('unique_industries', '')
    goals = row.get('unique_goals', '')
    actual_postal_codes = row.get('postal_codes', [])
    top_k = len(actual_postal_codes) if isinstance(actual_postal_codes, list) else 5

    result = recommendation_system.recommend_locations(
        advertiser_name=advertiser_name,
        industry=industry,
        goals=goals,
        top_k=top_k
    )

    return result['recommendations'] if result['recommendations'] else ''


In [48]:
import multiprocess as mp

if __name__ == '__main__':
    df_with_recommendations = final_preprocessed_df.copy()
    start_time = time.time()

    # Use all CPU cores
    with mp.Pool(processes=mp.cpu_count()) as pool:
        recommendations = pool.map(process_row, [row for _, row in df_with_recommendations.iterrows()])

    # Add the result to the DataFrame
    df_with_recommendations['recommended_postal_code'] = recommendations

    print(f"Time taken with multiprocessing: {time.time() - start_time:.2f} seconds")


Time taken with multiprocessing: 3.82 seconds


In [ ]:
import multiprocess as mp

if __name__ == '__main__':
    df_with_empty_goals_recommendations = customized_df.copy()
    start_time = time.time()

    # Use all CPU cores
    with mp.Pool(processes=mp.cpu_count()) as pool:
        recommendations = pool.map(process_row, [row for _, row in df_with_empty_goals_recommendations.iterrows()])

    # Add the result to the DataFrame
    df_with_empty_goals_recommendations['recommended_postal_code'] = recommendations

    print(f"Time taken with multiprocessing: {time.time() - start_time:.2f} seconds")
